# Chapter 08: Nonholonomic Motion Planning

Source orientation: printed pp. 355-394; PDF pp. 373-412. Chapter question: **How can Lie-bracket controllability be converted into actual steering commands?**

This notebook is an original, standalone teaching chapter. It uses the textbook for structure and mathematical orientation, but the explanations, code, and figures are created here. The chapter focus is sinusoidal steering, chained forms, optimal steering, BCH words, dynamic finger repositioning. The key objects are Brockett systems, sinusoidal controls, chained forms, Fourier steering, piecewise-constant bracket words, endpoint errors, and geometric phase for rolling contact.

Generated artifacts live under `artifacts/chapter-08` and are displayed both by code and in the static gallery. The final sanity cell checks the mathematical claims and artifact files so that the notebook remains auditable after reruns.


## Translation Guide

The chapter turns controllability into construction. Instead of only proving hidden directions exist, it designs inputs whose signed areas, harmonics, or bracket words produce desired endpoint motion. The computational translation used here is deliberately concrete: every named object becomes an array, graph, cone, trajectory, or map whose dimensions can be checked. That keeps the notebook independent from the PDF while preserving the chapter's mathematical route.

The main objects for this chapter are Brockett systems, sinusoidal controls, chained forms, Fourier steering, piecewise-constant bracket words, endpoint errors, and geometric phase for rolling contact. Read each one as a map between spaces. Ask what frame or chart supplies its coordinates, what operation changes those coordinates, and what quantity should remain invariant. The visuals in this notebook are chosen to make those questions inspectable rather than decorative.

The source pages are used only as orientation. Definitions and examples are paraphrased into course language, and all figures are generated from fresh code. When a visual resembles a textbook concept, it is a redraw or computational experiment rather than a page crop.


## Route Through The Chapter

We move in four passes. First, the notebook names the chapter's geometric objects and their engineering purpose. Second, it generates the visual sequence: brockett sinusoidal steering, chained form harmonics, geometric phase fingertip. Third, it runs a compact computational check that records the relevant residuals, ranks, endpoint errors, determinants, or signs. Fourth, it closes with an applied lab that asks the reader to change a parameter and explain what stayed invariant.

The important habit is to compare the visual artifact with the JSON check. If a cone claims feasibility, the feasibility check should agree. If a Jacobian claims usable motion, the task-space determinant or rank should agree. If a loop claims bracket displacement, the endpoint check should reveal it. The notebook is designed so those cross-checks are near each other.


In [ ]:
from pathlib import Path
import sys
import numpy as np

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the robotics course root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
TOPIC = "chapter-08"

from utils.artifacts import display_artifact, save_json
from utils.visuals import build_storyboard, storyboard_check_payload


In [ ]:
storyboard = {
  "label": "Chapter 08: Nonholonomic Motion Planning",
  "artifact_topic": "chapter-08",
  "source_span": "printed pp. 355-394; PDF pp. 373-412",
  "visual_sequence": [
    {
      "kind": "steering",
      "concept": "Brockett Sinusoidal Steering",
      "filename": "brockett-sinusoidal-steering.png",
      "observation": "area in controls creates bracket motion"
    },
    {
      "kind": "chained",
      "concept": "Chained Form Harmonics",
      "filename": "chained-form-harmonics.png",
      "observation": "harmonics rectify higher coordinates"
    },
    {
      "kind": "geometric_phase",
      "concept": "Geometric Phase Fingertip",
      "filename": "geometric-phase-fingertip.png",
      "observation": "rolling loop accumulates orientation"
    }
  ]
}

visual_results = build_storyboard(storyboard, ARTIFACT_ROOT, TOPIC)
visual_payload = storyboard_check_payload(storyboard, visual_results)
for item in visual_results:
    display_artifact(item["path"], width=720)
visual_payload


## Static Artifact Gallery

![Brockett Sinusoidal Steering](../../artifacts/chapter-08/figures/brockett-sinusoidal-steering.png)

*Inspection target:* area in controls creates bracket motion.

![Chained Form Harmonics](../../artifacts/chapter-08/figures/chained-form-harmonics.png)

*Inspection target:* harmonics rectify higher coordinates.

![Geometric Phase Fingertip](../../artifacts/chapter-08/figures/geometric-phase-fingertip.png)

*Inspection target:* rolling loop accumulates orientation.


## Worked Interpretation

The steering check integrates Brockett's system around a sinusoidal loop and compares the final bracket coordinate with the line integral around the visible control loop. The geometric-phase plot uses the same idea for rolling contact. This is the chapter's small worked example, not a full simulator. It is intentionally narrow enough that the relevant convention can be seen directly in code and broad enough to catch the common failure mode.

The visual sequence and the executable check use compatible parameters whenever possible. The point is to avoid a split course where the images tell one story and the numbers tell another. When extending this notebook, reuse that pattern: pick one invariant, draw the geometry that exposes it, then save a check payload that would fail if the convention were reversed or the rank condition were lost.

**Pitfall to watch:** A closed loop in the directly actuated coordinates is not a failed maneuver. For nonholonomic systems, the loop is often the point: its area or ordered word is what produces drift in the bracket coordinate. This pitfall is why the notebook avoids silent coordinate conventions. Names, frames, dimensions, and signs are part of the teaching surface, not implementation trivia.


In [ ]:
from utils.nonholonomic import integrate_brockett, sinusoid_controls

t, controls = sinusoid_controls(steps=720, amplitude=0.4)
dt = 2 * np.pi / len(t)
path = integrate_brockett(controls, dt=dt)
dx = np.diff(path[:, 0])
dy = np.diff(path[:, 1])
line_integral = float(np.sum(path[:-1, 0] * dy - path[:-1, 1] * dx))
endpoint = path[-1]
check_payload = {
    "endpoint": endpoint.round(6).tolist(),
    "bracket_coordinate": float(endpoint[2]),
    "control_loop_line_integral": line_integral,
    "line_integral_error": float(abs(endpoint[2] - line_integral)),
}
assert abs(endpoint[0]) < 0.01
assert abs(endpoint[1]) < 0.01
assert check_payload["bracket_coordinate"] > 0
assert check_payload["line_integral_error"] < 1e-12
check_payload


## Applied Lab

Lab prompt: Steer Brockett's system with a sinusoidal loop and compare the endpoint with signed area.

Before running the lab, make a prediction in three parts: which geometric object is being changed, which representation will show the change most clearly, and which scalar check should move in the same direction. After running it, compare the prediction with the saved JSON payload under `artifacts/chapter-08/labs`.

Use the pitfall above as a diagnostic. If the visual and scalar check disagree, inspect the frame convention, normalization, rank threshold, sign convention, or parameter range. The best result is not merely a green check; it is a short explanation of why the check protects the chapter's main idea.


In [ ]:
lab_notes = {
    "lab": "Steer Brockett's system with a sinusoidal loop and compare the endpoint with signed area.",
    "source_orientation": "printed pp. 355-394; PDF pp. 373-412",
    "artifact_topic": TOPIC,
    "reader_task": "Change one parameter, rerun the visual cell, and explain which invariant did or did not change.",
}
lab_path = save_json(lab_notes, TOPIC, "labs", "applied-lab.json", root=ARTIFACT_ROOT)
display_artifact(lab_path)


In [ ]:
from pathlib import Path

visual_paths = [Path(item["path"]) for item in visual_results]
assert visual_payload["assertions"]["has_multiple_visuals"]
assert visual_payload["assertions"]["all_visuals_nonblank"]
assert all(path.exists() and path.stat().st_size > 1000 for path in visual_paths)

final_sanity = {
    "visual_payload": visual_payload,
    "checks": check_payload,
    "visual_artifact_count": len(visual_paths),
    "visual_artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in visual_paths],
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json", root=ARTIFACT_ROOT)
display_artifact(sanity_path)
final_sanity


## Practice And Inspection Notes

Use this chapter as a small laboratory, not as a static summary. The source span is printed pp. 355-394 and PDF pp. 373-412, but the working material is the notebook itself: the generated artifacts, the executable check payload, and the applied lab. Start by reading the chapter question again: **How can Lie-bracket controllability be converted into actual steering commands?** Then identify which part of the visual sequence gives the most direct answer. For this chapter the focus is sinusoidal steering, chained forms, optimal steering, BCH words, dynamic finger repositioning, so the useful inspection targets are not generic screenshots; they are the ranks, cones, motions, frames, phase loops, energy shapes, or dependency arrows that make that focus concrete.

The `brockett sinusoidal steering` artifact uses the `steering` visual family; inspect it for area in controls creates bracket motion. The `chained form harmonics` artifact uses the `chained` visual family; inspect it for harmonics rectify higher coordinates. The `geometric phase fingertip` artifact uses the `geometric_phase` visual family; inspect it for rolling loop accumulates orientation.

After viewing the artifacts, rerun the computational check cell and read the keys in `check_payload` as a miniature rubric. Each key should protect one concept from a plausible robotics mistake. A determinant or rank protects a singularity claim. A residual protects an equation or closure condition. A monotonicity flag protects a scale-law interpretation. An endpoint error protects a steering construction. A power-invariance error protects a frame transformation. If a check is near zero, ask why zero is the right target; if a check is positive, ask what physical or geometric margin it measures.

Try three variations. First, make a small parameter change that should preserve the chapter's main invariant, such as a coordinate-frame change, a harmless redraw scale, or a sample density increase. Second, make a parameter change that should stress the model, such as a near-singular joint pose, lower friction coefficient, larger controller delay, smaller bracket loop, or weaker tendon tension. Third, make a convention change deliberately, such as reversing a normal or swapping a body/spatial interpretation, and predict which check should fail. These variations turn the notebook from a solved example into a diagnostic tool.

When writing your own notes, use this sentence pattern: "The artifact shows ..., the computation checks ..., and the invariant is ... ." That pattern is intentionally repetitive because it catches vague understanding. A reader who can fill in those three blanks for this chapter can usually transfer the idea to a different mechanism, contact model, or control task without reopening the textbook.


## Takeaways

- Motion planning for nonholonomic systems is constructive Lie bracket bookkeeping.
- Sinusoids and chained forms create endpoint motion by coordinating phases, not by moving one coordinate at a time.
- Geometric phase is accumulated history; the loop matters even when the visible contact returns.
- Revisit the saved artifacts after changing parameters; the strongest learning usually comes from explaining why the invariant survived.
